In [1]:
import numpy as np
import scipy.optimize as opt

In [2]:
class SVM:
    def __init__(self, C=1, kernel='linear', gamma=1, degree=3, r=1, epsilon=0.1,  max_iter=200, task='classification', multi_class='ovo'):
        self.C = C
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.r = r
        self.epsilon = epsilon
        self.max_iter = max_iter
        self.task = task
        self.multi_class = multi_class
        self.X_train = None
        self.K_train = None
        self.classes = None
        self.classifiers = None
        self.beta = None
        self.b = None

    def kernel_type(self, x1, x2):
        if self.kernel == 'linear':
            return float(np.dot(x1, x2))
        if self.kernel == 'poly':
            return float((self.gamma * np.dot(x1, x2) + 1) ** self.degree)
        if self.kernel == 'rbf':
            return float(np.exp(-self.gamma * np.sum((x1 - x2) ** 2)))
        if self.kernel == 'sigmoid':
            return np.tanh(self.gamma * np.dot(x1, x2) + self.r)
        raise ValueError

    def kernel_matrix(self, X, Y=None):
        if Y is None:
            n = X.shape[0]
            K = np.zeros((n, n))
            for i in range(n):
                for j in range(i, n):
                    val = self.kernel_type(X[i], X[j])
                    K[i, j] = val
                    K[j, i] = val
            return K
        else:
            n_x = X.shape[0]
            n_y = Y.shape[0]
            K = np.zeros((n_x, n_y))
            for i in range(n_x):
                for j in range(n_y):
                    K[i, j] = self.kernel_type(X[i], Y[j])
            return K

    def smo_classification(self, y, K):
        n = len(y)
        Q = np.outer(y, y) * K

        def objective(alpha):
            return 0.5 * alpha @ Q @ alpha - np.sum(alpha)

        def jacobian(alpha):
            return Q @ alpha - np.ones(n)

        cons = {'type': 'eq', 'fun': lambda alpha: np.dot(alpha, y)}
        bounds = [(0, self.C) for _ in range(n)]
        res = opt.minimize(objective, np.zeros(n), jac=jacobian, bounds=bounds, constraints=cons, method='SLSQP', tol=self.tol, options={'maxiter': self.max_iter})
        alpha = res.x
        sv = (alpha > 1e-5) & (alpha < self.C - 1e-5)
        if np.any(sv):
            b = np.mean(y[sv] - (alpha * y) @ K[:, sv])
        else:
            b = 0.0
        return alpha, b

    def smo_regression(self, y, K):
        n = len(y)
        eps = self.epsilon
        C = self.C

        def objective(beta):
            return 0.5 * beta @ K @ beta + eps * np.sum(np.abs(beta)) - np.dot(y, beta)

        def jacobian(beta):
            return K @ beta + eps * np.sign(beta) - y

        cons = {'type': 'eq', 'fun': lambda beta: np.sum(beta)}
        bounds = [(-C, C) for _ in range(n)]
        res = opt.minimize(objective, np.zeros(n), jac=jacobian, bounds=bounds, constraints=cons, method='SLSQP', tol=self.tol, options={'maxiter': self.max_iter})
        beta = res.x
        b = self.svr_compute_b(beta, y, K, eps)
        return beta, b

    def fit_classification(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.X_train = X
        self.K_train = self.kernel_matrix(X)
        self.classes = np.unique(y)
        self.classifiers = []
        if self.multi_class == 'ovr':
            for cls in self.classes:
                y_bin = np.where(y == cls, 1.0, -1.0)
                alpha, b = self.smo_classification(y_bin, self.K_train)
                coefs = alpha * y_bin
                self.classifiers.append((coefs, b))
        elif self.multi_class == 'ovo':
            n_classes = len(self.classes)
            for i in range(n_classes):
                for j in range(i + 1, n_classes):
                    cls_i = self.classes[i]
                    cls_j = self.classes[j]
                    idx = np.where((y == cls_i) | (y == cls_j))[0]

                    y_pair = np.where(y[idx] == cls_i, 1.0, -1.0)
                    K_pair = self.K_train[np.ix_(idx, idx)]

                    alpha, b = self.smo_classification(y_pair, K_pair)
                    coefs = alpha * y_pair
                    self.classifiers.append((cls_i, cls_j, idx, coefs, b))

    def predict_classification(self, X):
        K_test = self.kernel_matrix(X, self.X_train)
        if self.multi_class == 'ovr':
            scores = np.zeros((X.shape[0], len(self.classes)))
            for k, (coefs, b) in enumerate(self.classifiers):
                scores[:, k] = K_test @ coefs + b
            return self.classes[np.argmax(scores, axis=1)]
        elif self.multi_class == 'ovo':
            votes = np.zeros((X.shape[0], len(self.classes)))
            for cls_i, cls_j, idx, coefs, b in self.classifiers:
                K_pair_test = K_test[:, idx]
                scores = K_pair_test @ coefs + b
                i_idx = np.where(self.classes == cls_i)[0][0]
                j_idx = np.where(self.classes == cls_j)[0][0]
                for m in range(X.shape[0]):
                    if scores[m] > 0:
                        votes[m, i_idx] += 1
                    else:
                        votes[m, j_idx] += 1
            return self.classes[np.argmax(votes, axis=1)]

    def fit_regression(self, X, y):
        self.X_train = X
        self.K_train = self.kernel_matrix(X)
        self.beta, self.b = self.smo_regression(y, self.K_train)

    def predict_regression(self, X):
        K_test = self.kernel_matrix(X, self.X_train)
        return K_test @ self.beta + self.b

    def fit(self, X, y):
        if self.task == 'classification':
            self.fit_classification(X, y)
        elif self.task == 'regression':
            self.fit_regression(X, y)
        return self

    def predict(self, X):
        if self.task == 'classification':
            return self.predict_classification(X)
        return self.predict_regression(X)

    def svr_compute_b(self, beta, y, K, eps):
        K_beta = K @ beta
        candidates = []
        for i in range(len(beta)):
            if 0 < abs(beta[i]) < self.C:
                candidates.append(y[i] - K_beta[i] - eps * np.sign(beta[i]))
        if candidates:
            return float(np.mean(candidates))
        return 0

In [3]:
from sklearn.datasets import make_blobs, make_moons, make_regression
from sklearn.metrics import accuracy_score, r2_score

def run_tests():
    print("--- ЗАПУСК ТЕСТОВ ---")

    # ==========================================
    # ТЕСТ 1: Бинарная линейная классификация
    # ==========================================
    print("\n1. Бинарная линейная классификация (kernel='linear')")
    X, y = make_blobs(n_samples=50, centers=2, random_state=42, cluster_std=0.5)
    model = SVM(C=1, kernel='linear', task='classification')
    model.fit(X, y)
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    print(f"   Точность: {acc:.4f} (Ожидалось > 0.95)")
    assert acc > 0.95, "Тест 1 провален!"

    # ==========================================
    # ТЕСТ 2: Многоклассовая классификация (OvO)
    # ==========================================
    print("\n2. Многоклассовая классификация (kernel='linear', multi_class='ovo')")
    X, y = make_blobs(n_samples=90, centers=3, random_state=42, cluster_std=0.5)
    model = SVM(C=1, kernel='linear', task='classification', multi_class='ovo')
    model.fit(X, y)
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    print(f"   Точность: {acc:.4f} (Ожидалось > 0.95)")
    assert acc > 0.95, "Тест 2 провален!"

    # ==========================================
    # ТЕСТ 3: Многоклассовая классификация (OvR)
    # ==========================================
    print("\n3. Многоклассовая классификация (kernel='linear', multi_class='ovr')")
    model = SVM(C=1, kernel='linear', task='classification', multi_class='ovr')
    model.fit(X, y)
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    print(f"   Точность: {acc:.4f} (Ожидалось > 0.95)")
    assert acc > 0.95, "Тест 3 провален!"

    # ==========================================
    # ТЕСТ 4: Нелинейная классификация (RBF)
    # ==========================================
    print("\n4. Нелинейная классификация (kernel='rbf')")
    X, y = make_moons(n_samples=100, noise=0.1, random_state=42)
    model = SVM(C=1, kernel='rbf', gamma=1.0, task='classification')
    model.fit(X, y)
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    print(f"   Точность: {acc:.4f} (Ожидалось > 0.95)")
    assert acc > 0.95, "Тест 4 провален!"

    # ==========================================
    # ТЕСТ 5: Регрессия (SVR)
    # ==========================================
    print("\n5. Регрессия (task='regression', kernel='rbf')")
    # Генерируем нелинейные данные (синусоида)
    X = np.sort(5 * np.random.rand(80, 1), axis=0)
    y = np.sin(X).ravel()
    y[::5] += 0.5 * (0.5 - np.random.rand(16)) # Добавляем шум

    model = SVM(C=10, kernel='rbf', gamma=1.0, epsilon=0.1, task='regression')
    model.fit(X, y)
    preds = model.predict(X)
    r2 = r2_score(y, preds)
    print(f"   R-квадрат: {r2:.4f} (Ожидалось > 0.90)")
    assert r2 > 0.90, "Тест 5 провален!"

    print("\n--- ВСЕ ТЕСТЫ УСПЕШНО ПРОЙДЕНЫ! ---")

if __name__ == "__main__":
    run_tests()

--- ЗАПУСК ТЕСТОВ ---

1. Бинарная линейная классификация (kernel='linear')
   Точность: 1.0000 (Ожидалось > 0.95)

2. Многоклассовая классификация (kernel='linear', multi_class='ovo')
   Точность: 1.0000 (Ожидалось > 0.95)

3. Многоклассовая классификация (kernel='linear', multi_class='ovr')
   Точность: 1.0000 (Ожидалось > 0.95)

4. Нелинейная классификация (kernel='rbf')
   Точность: 0.9800 (Ожидалось > 0.95)

5. Регрессия (task='regression', kernel='rbf')
   R-квадрат: 0.9900 (Ожидалось > 0.90)

--- ВСЕ ТЕСТЫ УСПЕШНО ПРОЙДЕНЫ! ---
